In [12]:

# Step 1: Import Libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Step 2: Load Dataset

df = pd.read_csv("Telecom_Customer_Churn.csv")


# Step 3: Explore Dataset

print("First 5 Records:")
print(df.head())

print("\nDataset Information:")
df.info()

print("\nDataset Shape:")
print(df.shape)

print("\nStatistical Summary:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())


# Step 4: Handle Missing Values

df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'], errors='coerce'
)

df['MonthlyCharges'] = df['MonthlyCharges'].fillna(
    df['MonthlyCharges'].median()
)

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['TotalCharges'].median()
)

print("\nMissing Values After Handling:")
print(df.isnull().sum())


# Step 5: Remove Duplicate Records

df.drop_duplicates(inplace=True)

print("\nDuplicates Removed Successfully!")
print("Dataset Shape:", df.shape)


# Step 6: Handle Inconsistent Data

# Remove extra spaces
for col in df.select_dtypes(include=['object', 'str']).columns:
    df[col] = df[col].str.strip()

# Standardize Gender
df['gender'] = df['gender'].str.lower()

# Standardize Contract
df['Contract'] = df['Contract'].str.lower()

# Standardize Churn
df['Churn'] = df['Churn'].str.capitalize()

print("\nInconsistent Data Standardized!")

print("\nGender:", df['gender'].unique())
print("Contract:", df['Contract'].unique())
print("Churn:", df['Churn'].unique())


# Step 7: Convert Columns to Correct Data Types

df['customerID'] = pd.to_numeric(
    df['customerID'], errors='coerce'
)

df['tenure'] = pd.to_numeric(df['tenure'])

df['MonthlyCharges'] = pd.to_numeric(
    df['MonthlyCharges'], errors='coerce'
)

df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'], errors='coerce'
)

print("\nData Types:")
print(df.dtypes)


# Step 8: Identify and Handle Outliers

columns = ['tenure', 'MonthlyCharges', 'TotalCharges']

for col in columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Handle outliers using IQR clipping
    df[col] = df[col].clip(lower, upper)

print("\nOutliers Handled Successfully!")


# Step 9: Feature Engineering

# Average Monthly Charge
df['AvgCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

# Create Tenure Groups
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 72],
    labels=['New', 'Medium', 'Long', 'Very Long']
)

print("\nNew Features Created!")

print(df[['AvgCharge', 'TenureGroup']].head())


# Step 10: Normalize / Scale Data

scaler = StandardScaler()

columns = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'AvgCharge'
]

df[columns] = scaler.fit_transform(df[columns])

print("\nData Scaling Completed!")


# Step 11: Split Dataset into Training and Testing Sets

# Convert Churn into numerical values
df['Churn'] = df['Churn'].map({
    'Yes': 1,
    'No': 0
})

# Encode categorical columns
df = pd.get_dummies(
    df,
    columns=['gender', 'Contract', 'TenureGroup'],
    drop_first=True,
    dtype=int
)

# Separate features and target
X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Data:", X_train.shape)
print("Testing Data:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)


# Step 12: Export Cleaned Dataset

df.to_csv("Telecom_Cleaned1.csv", index=False)

print("\nCleaned Dataset Saved Successfully!")



First 5 Records:
   customerID  gender  tenure  MonthlyCharges        Contract TotalCharges  \
0           0    male      50           96.09        Two year                
1           1    male      47           52.80        Two year       2481.6   
2           2  FEMALE      10          106.75  Month-to-month       1067.5   
3           3  FEMALE      22           57.02  Month-to-month      1254.44   
4           4    Male      24           25.43  Month-to-month       610.32   

  Churn  
0    No  
1    No  
2    No  
3   Yes  
4    No  

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customerID      1015 non-null   int64  
 1   gender          1015 non-null   str    
 2   tenure          1015 non-null   int64  
 3   MonthlyCharges  994 non-null    float64
 4   Contract        1015 non-null   str    
 5   TotalCharges    1